# ML Model Comparison

models need to be trained first - run train_models.ipynb

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob

warnings.filterwarnings("ignore")

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(DATA_DIR, "models")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

print(f"BASE_DIR:    {BASE_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"MODELS_DIR:  {MODELS_DIR}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

## 1. Load Trained Predictor

In [ ]:
import joblib

def load_predictor_stats(filename):
    path = os.path.join(MODELS_DIR, filename)
    if not os.path.exists(path):
        print(f"[WARN] Not found: {filename}")
        return None, []
    p = joblib.load(path)
    ts = p["training_stats"]["result"]
    names = list(ts.get("detailed_metrics", {}).keys())
    return ts, names

training_stats, model_names = load_predictor_stats("universal_predictor.pkl")
training_stats_odds, model_names_odds = load_predictor_stats("universal_predictor_with_odds.pkl")

print("Model WITHOUT odds:")
if training_stats:
    for name in model_names:
        m = training_stats["detailed_metrics"][name]
        print(f"  {name:25s} acc={m.get('accuracy', 0):.3f}  f1={m.get('f1', 0):.3f}")
    print(f"  Features: {training_stats.get('features')}  |  train/test: {training_stats.get('train_matches')}/{training_stats.get('test_matches')}")

print("\nModel WITH odds:")
if training_stats_odds:
    for name in model_names_odds:
        m = training_stats_odds["detailed_metrics"][name]
        print(f"  {name:25s} acc={m.get('accuracy', 0):.3f}  f1={m.get('f1', 0):.3f}")
    print(f"  Features: {training_stats_odds.get('features')}  |  train/test: {training_stats_odds.get('train_matches')}/{training_stats_odds.get('test_matches')}")

## 2. Load Prediction Reports

In [ ]:
def load_all_reports(reports_dir):
    records = []
    pattern = os.path.join(reports_dir, "*", "predictions_finished.json")
    for fpath in sorted(glob(pattern)):
        date = os.path.basename(os.path.dirname(fpath))
        try:
            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"[WARN] {fpath}: {e}")
            continue
        for match in data.get("matches", []):
            actual = match.get("actual_result")
            if actual not in ("HOME", "DRAW", "AWAY"):
                continue
            preds = match.get("predictions", {})  # result predictions only
            row = {
                "date": date,
                "competition": match.get("league", ""),
                "home_team": match.get("home_team", ""),
                "away_team": match.get("away_team", ""),
                "actual": actual,
                "actual_score": match.get("actual_score"),
            }
            for model, pred_data in preds.items():
                if isinstance(pred_data, dict):
                    row[f"pred_{model}"] = pred_data.get("prediction")
                    conf = pred_data.get("confidence")
                    row[f"conf_{model}"] = conf / 100.0 if conf is not None else None
                    probs = pred_data.get("probabilities", {})
                    row[f"prob_HOME_{model}"] = probs.get("HOME")
                    row[f"prob_DRAW_{model}"] = probs.get("DRAW")
                    row[f"prob_AWAY_{model}"] = probs.get("AWAY")
            records.append(row)
    return pd.DataFrame(records)

df = load_all_reports(REPORTS_DIR)
print(f"Total finished matches in reports: {len(df)}")
if len(df) > 0:
    print(f"Date range: {df['date'].min()} -> {df['date'].max()}")
    print(f"Actual result distribution:")
    print(df["actual"].value_counts().to_string())
else:
    print("No finished matches found!")

## 3. Accuracy: Test Set vs Live Predictions

## 3b. Bookmaker Odds Experiment

two separate models - one without odds (all matches), one only where odds were available

In [ ]:
if training_stats and training_stats_odds:
    rows = []
    for name in model_names:
        m_no  = training_stats.get("detailed_metrics", {}).get(name, {})
        m_yes = training_stats_odds.get("detailed_metrics", {}).get(name, {})
        rows.append({
            "Model": name,
            "Acc (no odds)":   round(m_no.get("accuracy", 0), 4),
            "F1 (no odds)":    round(m_no.get("f1", 0), 4),
            "Acc (with odds)": round(m_yes.get("accuracy", 0), 4),
            "F1 (with odds)":  round(m_yes.get("f1", 0), 4),
        })

    cmp_df = pd.DataFrame(rows).sort_values("Acc (no odds)", ascending=False)
    cmp_df["Acc diff"] = (cmp_df["Acc (with odds)"] - cmp_df["Acc (no odds)"]).round(4)
    cmp_df["F1 diff"]  = (cmp_df["F1 (with odds)"]  - cmp_df["F1 (no odds)"]).round(4)
    print(cmp_df.to_string(index=False))

    x = np.arange(len(cmp_df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w/2, cmp_df["Acc (no odds)"],   w, label="Without odds", color="steelblue")
    ax.bar(x + w/2, cmp_df["Acc (with odds)"], w, label="With odds",    color="tomato")
    ax.set_xticks(x)
    ax.set_xticklabels(cmp_df["Model"], rotation=30, ha="right")
    ax.set_ylabel("Test Accuracy")
    ax.set_title("Impact of Bookmaker Odds on Model Accuracy")
    ax.legend()
    ax.axhline(1/3, color="gray", linestyle="--", alpha=0.5)
    ax.set_ylim(0.2, 0.75)
    plt.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(9, 4))
    colors = ["green" if v >= 0 else "red" for v in cmp_df["Acc diff"]]
    ax2.bar(cmp_df["Model"], cmp_df["Acc diff"], color=colors)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_ylabel("Accuracy diff (with - without odds)")
    ax2.set_title("Gain from Including Bookmaker Odds")
    ax2.set_xticklabels(cmp_df["Model"], rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("universal_predictor_with_odds.pkl not found - run train_models.ipynb first.")

In [ ]:
rows = []
for model in model_names:
    m = training_stats["detailed_metrics"][model]
    pred_col = f"pred_{model}"
    if pred_col not in df.columns:
        continue
    sub = df[df[pred_col].notna()]
    live_acc = (sub[pred_col] == sub["actual"]).mean() if len(sub) > 0 else np.nan
    rows.append({
        "Model": model,
        "Test Accuracy": round(m.get("accuracy", 0), 4),
        "Test F1": round(m.get("f1", 0), 4),
        "Live Accuracy": round(live_acc, 4) if not np.isnan(live_acc) else None,
        "Live Matches": len(sub),
    })

if not rows:
    print("No matching pred_* columns found.")
    print("model_names:", model_names)
    print("pred cols in df:", [c for c in df.columns if c.startswith("pred_")][:5])
else:
    acc_df = pd.DataFrame(rows).sort_values("Test Accuracy", ascending=False)
    print(acc_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(acc_df))
    w = 0.35
    ax.bar(x - w/2, acc_df["Test Accuracy"], w, label="Test Accuracy", color="steelblue")
    ax.bar(x + w/2, acc_df["Live Accuracy"].fillna(0), w, label="Live Accuracy", color="tomato")
    ax.set_xticks(x)
    ax.set_xticklabels(acc_df["Model"], rotation=30, ha="right")
    ax.set_ylabel("Accuracy")
    ax.set_title("Model Accuracy: Test Set vs Live Predictions")
    ax.legend()
    ax.axhline(1/3, color="gray", linestyle="--", alpha=0.5)
    ax.set_ylim(0.2, 0.75)
    plt.tight_layout()
    plt.show()

## 4. Per-League Accuracy

In [ ]:
pred_cols = [c for c in df.columns if c.startswith("pred_")]
model_names_live = [c.replace("pred_", "") for c in pred_cols if c.replace("pred_", "") in model_names]

league_rows = []
for comp, grp in df.groupby("competition"):
    if len(grp) < 5:
        continue
    row = {"Competition": comp, "Matches": len(grp)}
    for model in model_names_live:
        pc = f"pred_{model}"
        if pc in grp.columns:
            sub = grp[grp[pc].notna()]
            row[model] = round((sub[pc] == sub["actual"]).mean(), 3) if len(sub) > 0 else None
    league_rows.append(row)

if league_rows:
    league_df = pd.DataFrame(league_rows).sort_values("Matches", ascending=False)
    print(league_df.to_string(index=False))
else:
    print("Not enough data per league yet (<5 matches each).")

## 5. Per-Result-Type Accuracy (H / D / A)

In [ ]:
result_rows = []
for result in ["HOME", "DRAW", "AWAY"]:
    sub = df[df["actual"] == result]
    row = {"Result": result, "Matches": len(sub)}
    for model in model_names_live:
        pc = f"pred_{model}"
        if pc in sub.columns:
            s = sub[sub[pc].notna()]
            row[model] = round((s[pc] == s["actual"]).mean(), 3) if len(s) > 0 else None
    result_rows.append(row)

result_df = pd.DataFrame(result_rows)
print(result_df.to_string(index=False))

if model_names_live:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(3)
    w = 0.8 / len(model_names_live)
    for i, model in enumerate(model_names_live):
        vals = []
        for r in ["HOME", "DRAW", "AWAY"]:
            val = result_df.set_index("Result").loc[r, model] if model in result_df.columns else 0
            vals.append(val if pd.notna(val) else 0)
        ax.bar(x + i * w - 0.4 + w/2, vals, w, label=model)
    ax.set_xticks(x)
    ax.set_xticklabels(["Home Win", "Draw", "Away Win"])
    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy by Result Type")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. Calibration: Confidence vs Actual Accuracy

In [ ]:
bins = np.arange(0.3, 1.05, 0.1)
bin_labels = [f"{b:.1f}-{b+0.1:.1f}" for b in bins[:-1]]

n_models = min(len(model_names_live), 8)
rows_count = max((n_models + 3) // 4, 1)
fig, axes = plt.subplots(rows_count, 4, figsize=(16, 4 * rows_count), sharey=True)
axes = np.array(axes).flatten()

for idx, model in enumerate(model_names_live[:8]):
    pc = f"pred_{model}"
    cc = f"conf_{model}"
    if pc not in df.columns or cc not in df.columns or idx >= len(axes):
        continue
    sub = df[df[pc].notna() & df[cc].notna()].copy()
    sub["correct"] = (sub[pc] == sub["actual"]).astype(int)
    sub["bin"] = pd.cut(sub[cc], bins=bins, labels=bin_labels, right=False)
    calib = sub.groupby("bin", observed=True).agg(acc=("correct", "mean"), cnt=("correct", "count")).reset_index()

    ax = axes[idx]
    ax.bar(range(len(calib)), calib["acc"], color="steelblue", alpha=0.7)
    mid_vals = [float(str(b).split("-")[0]) + 0.05 for b in calib["bin"]]
    ax.plot(range(len(calib)), mid_vals, "r--")
    ax.set_xticks(range(len(calib)))
    ax.set_xticklabels(calib["bin"], rotation=45, ha="right", fontsize=7)
    ax.set_title(model, fontsize=9)
    ax.set_ylim(0, 1)
    for j, row in calib.iterrows():
        ax.text(j, row["acc"] + 0.02, str(int(row["cnt"])), ha="center", fontsize=7, color="gray")

for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

fig.suptitle("Calibration: Confidence Bin vs Actual Accuracy", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Bookmaker Comparison

odds are not in reports - comparison is in section 3b above

## 8. Brier Score

lower is better - checks probability calibration, not just whether prediction was right

In [ ]:
def brier_score_model(df_sub, model):
    ph = f"prob_HOME_{model}"
    pd_ = f"prob_DRAW_{model}"
    pa = f"prob_AWAY_{model}"
    if not all(c in df_sub.columns for c in [ph, pd_, pa]):
        return None
    sub = df_sub[df_sub[ph].notna() & df_sub[pd_].notna() & df_sub[pa].notna()].copy()
    if len(sub) == 0:
        return None
    bs = (
        (sub[ph]/100 - (sub["actual"] == "HOME").astype(float))**2 +
        (sub[pd_]/100 - (sub["actual"] == "DRAW").astype(float))**2 +
        (sub[pa]/100 - (sub["actual"] == "AWAY").astype(float))**2
    ).mean()
    return round(float(bs), 4)

brier_rows = []
for model in model_names_live:
    bs = brier_score_model(df, model)
    n = len(df[df[f"pred_{model}"].notna()]) if f"pred_{model}" in df.columns else 0
    brier_rows.append({"Model": model, "Brier Score": bs, "Matches": n})

brier_df = pd.DataFrame(brier_rows).dropna(subset=["Brier Score"]).sort_values("Brier Score")
print(brier_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(brier_df["Model"], brier_df["Brier Score"], color="steelblue")
ax.set_ylabel("Brier Score (lower = better)")
ax.set_title("Multi-class Brier Score")
ax.set_xticklabels(brier_df["Model"], rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. Top Features by Mutual Information

In [ ]:
mi_scores = training_stats.get("mi_scores_top10")
feature_names = training_stats.get("feature_names", [])

if mi_scores:
    if isinstance(mi_scores, dict):
        pairs = sorted(mi_scores.items(), key=lambda x: -x[1])[:15]
    else:
        n = min(len(mi_scores), len(feature_names), 15)
        pairs = sorted(zip(feature_names[:n], mi_scores[:n]), key=lambda x: -x[1])

    names, scores = zip(*pairs)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(list(reversed(names)), list(reversed(scores)), color="teal")
    ax.set_xlabel("Mutual Information Score")
    ax.set_title(f"Top {len(pairs)} Features by Mutual Information (result model)")
    plt.tight_layout()
    plt.show()
else:
    print("No MI scores found in training_stats.")

## 10. Summary Table + Export to CSV

In [ ]:
summary_rows = []
for model in model_names:
    m = training_stats["detailed_metrics"][model]
    pc = f"pred_{model}"
    sub = df[df[pc].notna()] if pc in df.columns else pd.DataFrame()
    live_acc = (sub[pc] == sub["actual"]).mean() if len(sub) > 0 else None
    bs = brier_score_model(df, model)
    summary_rows.append({
        "Model": model,
        "Test Accuracy": round(m.get("accuracy", 0), 4),
        "Test F1": round(m.get("f1", 0), 4),
        "Live Accuracy": round(live_acc, 4) if live_acc is not None else None,
        "Live Matches": len(sub),
        "Brier Score": bs,
        "Train Time (s)": round(m.get("train_time_s", 0), 2),
        "Predict Time (ms)": round(m.get("predict_time_ms", 0), 3),
        "Memory (MB)": round(m.get("memory_mb", 0), 2),
        "Model Size (KB)": round(m.get("model_size_kb", 0), 1),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Test Accuracy", ascending=False)
print(summary_df.to_string(index=False))

out_path = os.path.join(MODELS_DIR, "comparison_summary.csv")
summary_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")